# 07 - Visualizacao da Segmentacao Binarizada

Le os artefatos analiticos gerados pelo notebook 06 e produz um relatorio visual focado na comparacao entre estrategias de binarizacao usando `iou`, `precision` e `recall`.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
plt.rcParams['figure.max_open_warning'] = 0
import pandas as pd
from IPython.display import display

root_dir = Path.cwd()
if not (root_dir / 'src').exists() and (root_dir.parent / 'src').exists():
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.io.path_resolver import PathResolver
from src.visualization import (
    PdfReportSection,
    plot_metric_bars_by_model,
    plot_metric_correlation_heatmap,
    plot_metric_distribution_by_model,
    plot_metric_scatter,
    plot_model_tag_interaction_heatmap,
    plot_pairwise_pvalue_heatmap,
    plot_simple_regression,
    plot_stability_bars,
    save_pdf_report,
)


## Carregamento das tabelas analiticas

O notebook 07 consome apenas os CSVs produzidos pelo notebook 06. Se algum arquivo ainda nao existir, execute o notebook 06 antes.


In [ ]:
metric_names = ['iou', 'precision', 'recall']
metric_pairs = [('iou', 'precision'), ('iou', 'recall'), ('precision', 'recall')]
interaction_metrics = ['iou', 'precision', 'recall']
path_resolver = PathResolver.from_config()
analysis_dir = Path(path_resolver.generated_dir) / 'analysis' / 'segmentacao_binarizada'
report_output_path = Path(path_resolver.generated_dir) / '07_visualizacao_segmentacao_binarizada.pdf'


def _load_csv(file_name):
    path = analysis_dir / file_name
    if not path.exists():
        raise FileNotFoundError(f'Arquivo analitico nao encontrado: {path}')
    return pd.read_csv(path)


df_base = _load_csv('base_analitica.csv')
df_resumo_estrategia = _load_csv('resumo_estrategia.csv')
df_resumo_modelo_estrategia = _load_csv('resumo_modelo_estrategia.csv')
df_estabilidade_modelo = _load_csv('estabilidade_modelo.csv')
df_estabilidade_modelo_estrategia = _load_csv('estabilidade_modelo_estrategia.csv')
df_testes_estrategia = _load_csv('testes_estrategia.csv')
df_testes_tag_estrategia = _load_csv('testes_tag_estrategia.csv')
df_interacoes_tag_estrategia = _load_csv('interacao_tag_estrategia.csv')

print(f'Base analitica: {len(df_base)} linhas')
print(f'Resumo por estrategia: {len(df_resumo_estrategia)} linhas')
print(f'Resumo por modelo + estrategia: {len(df_resumo_modelo_estrategia)} linhas')
print(f'Estabilidade por modelo: {len(df_estabilidade_modelo)} linhas')
print(f'Estabilidade por modelo + estrategia: {len(df_estabilidade_modelo_estrategia)} linhas')
print(f'Testes entre estrategias: {len(df_testes_estrategia)} linhas')
print(f'Testes de tag por estrategia: {len(df_testes_tag_estrategia)} linhas')
print(f'Interacoes tag x estrategia: {len(df_interacoes_tag_estrategia)} linhas')
print(f'PDF de saida: {report_output_path}')


## Comparacao geral entre estrategias

Este bloco resume cada metrica no nivel da estrategia de binarizacao. Ele responde primeiro a pergunta mais direta: qual estrategia tende a performar melhor no agregado, antes de abrir por modelo.


In [ ]:
texto_estrategias = 'Este bloco resume cada metrica no nivel da estrategia de binarizacao. Ele responde primeiro a pergunta mais direta: qual estrategia tende a performar melhor no agregado, antes de abrir por modelo.'
figures_estrategias = []

df_resumo_estrategia_plot = df_resumo_estrategia.rename(columns={'estrategia_binarizacao': 'nome_modelo'})

display(
    df_resumo_estrategia.pivot(index='estrategia_binarizacao', columns='metric_name', values='mean').sort_index()
)

for metric_name in metric_names:
    fig, _ = plot_metric_bars_by_model(df_resumo_estrategia_plot, metric_name)
    figures_estrategias.append(fig)
    plt.show()


## Distribuicoes por estrategia

Os boxplots ajudam a ver dispersao e outliers por estrategia, sem misturar isso com a identidade do modelo. Assim fica mais facil enxergar se uma estrategia ganha por centralidade ou apenas por poucos casos extremos.


In [ ]:
texto_distribuicao = 'Os boxplots ajudam a ver dispersao e outliers por estrategia, sem misturar isso com a identidade do modelo. Assim fica mais facil enxergar se uma estrategia ganha por centralidade ou apenas por poucos casos extremos.'
figures_distribuicao = []

df_base_por_estrategia = df_base.rename(columns={'estrategia_binarizacao': 'modelo'})

for metric_name in metric_names:
    fig, _ = plot_metric_distribution_by_model(df_base_por_estrategia, metric_name)
    figures_distribuicao.append(fig)
    plt.show()


## Estabilidade entre execucoes

A estabilidade agora e lida no nivel `modelo + estrategia`. Isso permite separar uma estrategia que e boa, mas sensivel a variacao entre execucoes, de outra que e um pouco inferior, porem mais repetivel.


In [ ]:
texto_estabilidade = 'A estabilidade agora e lida no nivel modelo + estrategia. Isso permite separar uma estrategia que e boa, mas sensivel a variacao entre execucoes, de outra que e um pouco inferior, porem mais repetivel.'
figures_estabilidade = []

df_estabilidade_plot = df_estabilidade_modelo_estrategia.copy()
df_estabilidade_plot['nome_modelo'] = (
    df_estabilidade_plot['modelo'].astype(str) + ' | ' + df_estabilidade_plot['estrategia_binarizacao'].astype(str)
)

display(df_estabilidade_modelo_estrategia.sort_values(['metric_name', 'cv_execucoes', 'modelo', 'estrategia_binarizacao']).head(30))

for metric_name in metric_names:
    fig, _ = plot_stability_bars(df_estabilidade_plot, metric_name)
    figures_estabilidade.append(fig)
    plt.show()


## Comparacao estatistica entre estrategias

Aqui entram os testes nao parametricos entre estrategias. O heatmap usa o `p_value_adjusted` dos testes pareados globais; quanto menor o valor, maior a evidencia de diferenca estatistica entre duas estrategias naquela metrica.


In [ ]:
texto_testes = 'Aqui entram os testes nao parametricos entre estrategias. O heatmap usa o p_value_adjusted dos testes pareados globais; quanto menor o valor, maior a evidencia de diferenca estatistica entre duas estrategias naquela metrica.'
figures_testes = []

df_testes_estrategia_global = df_testes_estrategia.loc[
    df_testes_estrategia['modelo_origem'] == '__global__'
].copy()

if df_testes_estrategia_global.empty:
    print('Nao ha testes globais entre estrategias disponiveis nos CSVs atuais.')
else:
    display(df_testes_estrategia_global.sort_values(['metric_name', 'comparison_scope', 'p_value_adjusted']).head(30))

    for metric_name in metric_names:
        fig, _ = plot_pairwise_pvalue_heatmap(df_testes_estrategia_global, metric_name)
        figures_testes.append(fig)
        plt.show()


## Analise bivariada e correlacao

Como `iou`, `precision` e `recall` podem contar historias diferentes, este bloco verifica associacao entre elas. Aqui cada ponto representa uma observacao binarizada, colorida pela estrategia.


In [ ]:
texto_correlacao = 'Como iou, precision e recall podem contar historias diferentes, este bloco verifica associacao entre elas. Aqui cada ponto representa uma observacao binarizada, colorida pela estrategia.'
figures_correlacao = []

display(df_base[metric_names].corr(method='pearson'))
display(df_base[metric_names].corr(method='spearman'))

for x_metric, y_metric in metric_pairs:
    fig, _ = plot_metric_scatter(df_base_por_estrategia, x_metric, y_metric)
    figures_correlacao.append(fig)
    plt.show()

for method in ['pearson', 'spearman']:
    fig, _ = plot_metric_correlation_heatmap(df_base, metric_names, method)
    figures_correlacao.append(fig)
    plt.show()


## Regressao simples

A regressao simples usa `num_tags_problema` como proxy de dificuldade. O objetivo aqui e medir se as estrategias binarizadas degradam de forma mais forte quando a imagem acumula problemas de recorte.


In [ ]:
texto_regressao = 'A regressao simples usa num_tags_problema como proxy de dificuldade. O objetivo aqui e medir se as estrategias binarizadas degradam de forma mais forte quando a imagem acumula problemas de recorte.'
figures_regressao = []

for metric_name in metric_names:
    fig, _ = plot_simple_regression(df_base_por_estrategia, 'num_tags_problema', metric_name)
    figures_regressao.append(fig)
    plt.show()


## Interacao entre estrategia e dificuldade

Este bloco resume como cada tag de problema desloca as metricas de cada estrategia. O heatmap usa `adjusted_delta_mean`: valores mais negativos sugerem piora maior associada a determinada dificuldade.


In [ ]:
texto_interacoes = 'Este bloco resume como cada tag de problema desloca as metricas de cada estrategia. O heatmap usa adjusted_delta_mean: valores mais negativos sugerem piora maior associada a determinada dificuldade.'
figures_interacoes = []

df_interacoes_plot = df_interacoes_tag_estrategia.rename(columns={'estrategia_binarizacao': 'nome_modelo'})

display(df_interacoes_tag_estrategia.sort_values(['metric_name', 'tag_name', 'adjusted_delta_mean']).head(30))

for metric_name in interaction_metrics:
    fig, _ = plot_model_tag_interaction_heatmap(df_interacoes_plot, metric_name)
    figures_interacoes.append(fig)
    plt.show()


## Leitura inicial

O notebook 07 prioriza uma leitura objetiva da segmentacao binarizada: comparacao geral entre estrategias, distribuicoes, estabilidade, significancia estatistica entre estrategias, relacao entre metricas e sensibilidade a dificuldade.


In [ ]:
report_sections = [
    PdfReportSection(
        heading='Carregamento das tabelas analiticas',
        body='O notebook 07 consome apenas os CSVs produzidos pelo notebook 06. Se algum arquivo ainda nao existir, execute o notebook 06 antes.',
    ),
    PdfReportSection(heading='Comparacao geral entre estrategias', body=texto_estrategias, figures=figures_estrategias),
    PdfReportSection(heading='Distribuicoes por estrategia', body=texto_distribuicao, figures=figures_distribuicao),
    PdfReportSection(heading='Estabilidade entre execucoes', body=texto_estabilidade, figures=figures_estabilidade),
    PdfReportSection(heading='Comparacao estatistica entre estrategias', body=texto_testes, figures=figures_testes),
    PdfReportSection(heading='Analise bivariada e correlacao', body=texto_correlacao, figures=figures_correlacao),
    PdfReportSection(heading='Regressao simples', body=texto_regressao, figures=figures_regressao),
    PdfReportSection(heading='Interacao entre estrategia e dificuldade', body=texto_interacoes, figures=figures_interacoes),
    PdfReportSection(
        heading='Leitura inicial',
        body='O notebook 07 prioriza uma leitura objetiva da segmentacao binarizada: comparacao geral entre estrategias, distribuicoes, estabilidade, significancia estatistica entre estrategias, relacao entre metricas e sensibilidade a dificuldade.',
    ),
]

pdf_path = save_pdf_report(
    output_path=report_output_path,
    sections=report_sections,
    report_title='07 - Visualizacao da Segmentacao Binarizada',
)

print(f'Relatorio PDF salvo em: {pdf_path}')
